In [1]:
import math
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from PIL import Image
from transformers import TrainerCallback
from unsloth import FastVisionModel 
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
import pandas as pd
from sklearn.model_selection import train_test_split
from torchsummary import summary


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-01 17:53:05.796746: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-01 17:53:05.949997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754038386.028951   90189 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754038386.054662   90189 cuda_blas

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[2025-08-01 17:53:14,213] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/cqilab/anaconda3/envs/llmfinetune/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/cqilab/anaconda3/envs/llmfinetune/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


ModuleNotFoundError: No module named 'torchsummary'

In [ ]:
!pip install torch-summary

In [ ]:
BASE_CSV = "Dataset/wigner_analysis_results_combined.csv"
CIR_CSV  = "case_study_circuit.csv"
ENT_CSV  = "case_study_entanglement.csv"

# ------------ load baseline dataset ------------


BEST_PROMPT = (
    "You are given a grayscale image representing a quantum optical state. "
    "Your task is to determine the type of the state (e.g., cat state, Fock state, "
    "coherent state, thermal state, random state etc.) as well as its key parameters "
    "(alpha/number of photons/density, number of qubits, and the linear space range). "
    "Please provide your answer in the format: "
    "\"<think>[THINKING PROCESS]</think> This is a [STATE TYPE] with [KEY parameters] "
    "equal to [VALUE], number of qubits equal to [N] in the linear space [LOW] to [HIGH].\" "
    "then extract your opinion on how you determine state, parameters, number of qubit "
    "from the image."
)

# ------------ format baseline ------------------
def csv_to_ft_lists(csv_path, test_frac=0.1, seed=42):
    df = pd.read_csv(csv_path).sample(frac=1, random_state=seed).reset_index(drop=True)
    train_df, test_df = train_test_split(df, test_size=test_frac, random_state=seed)
    return train_df.to_dict("records"), test_df.to_dict("records")

# def build_ft_list(df_slice):
#     return [
#         {"image": row["image"], "input": BEST_PROMPT, "output": row["ground_truth"]}
#         for _, row in df_slice.iterrows()
#     ]

# fine_tune_data = build_ft_list(base_train)
# eval_data      = build_ft_list(base_test.iloc[:3])  # keep 2-3 samples for quick eval

# ------------ load extra case-study datasets ---
circuit_train, circuit_eval = csv_to_ft_lists(CIR_CSV)
entangle_train, entangle_eval = csv_to_ft_lists(ENT_CSV)
circuit_eval   = circuit_eval[:3]
entangle_eval  = entangle_eval[:3]

In [ ]:
class FinetuneQwenVL:
    def __init__(self, 
                 data,
                 eval_data,
                 epochs=1, 
                 learning_rate=1e-4,
                 warmup_ratio=0.1,
                 gradient_accumulation_steps=64,
                 optim="adamw_torch",
                 model_id="unsloth/Qwen2-VL-7B-Instruct", 
                 peft_r=8,
                 peft_alpha=16,
                 peft_dropout=0.05,
                ):
        """
        Args:
            data: a list of dicts for training
            eval_data: a list of dicts for evaluating (2-3 samples for quick tests every epoch)
        """
        self.epochs = epochs
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model_id = model_id

        # 1) Load base model and tokenizer
        self.base_model, self.tokenizer = FastVisionModel.from_pretrained(
            model_name = self.model_id,
            load_in_4bit = False,
            use_gradient_checkpointing = "unsloth",
        )
        
        # 2) Wrap with PEFT / LoRA
        self.model = FastVisionModel.get_peft_model(
            self.base_model,
            finetune_vision_layers     = True, # set True if you want vision layers updated
            finetune_language_layers   = True, # set True if you want language layers updated
            finetune_attention_modules = True,
            finetune_mlp_modules       = True,
            r = peft_r,
            lora_alpha = peft_alpha,
            lora_dropout = peft_dropout,
            bias = "none",
            random_state = 3407,
            use_rslora = False,
            loftq_config = None
        )
        
        self.learning_rate = learning_rate
        self.warmup_ratio = warmup_ratio
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.optim = optim
        self.data = data
        self.eval_data = eval_data

    def format_data(self, row):
        image_path = row["image"]
        input_text = row['input']
        output_text = row['output']
        
        try:
            image = Image.open(image_path).convert("RGB")
            # If needed, you can also resize or transform:
            image = image.resize((1000, 600))
        except Exception as e:
            raise FileNotFoundError(
                f"Unable to load image at path: {image_path}. Error: {e}"
            )

        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": input_text,
                        },
                        {
                            "type": "image",
                            "image": image,  
                        }
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {
                            "type": "text",
                            "text": output_text,
                        }
                    ],
                },
            ],
        }
        
    def format_data_multiturn(self, row):
        img_path = row["image"]
        try:
            img = Image.open(img_path).convert("RGB").resize((1000, 600))
        except Exception as e:
            raise FileNotFoundError(f"Cannot open {img_path}: {e}")

        messages   = []
        image_sent = False
        turn       = 1

        while True:
            in_key  = f"input_{turn}"
            out_key = f"output_{turn}"
            if in_key not in row or out_key not in row:
                break

            user_text      = row[in_key]
            assistant_text = row[out_key]

            if user_text is None or assistant_text is None:
                break
            if isinstance(user_text, float) and math.isnan(user_text):
                break
            if isinstance(assistant_text, float) and math.isnan(assistant_text):
                break
            if str(user_text).strip() == "" and str(assistant_text).strip() == "":
                break

            # ----- user message -----
            user_content = []
            if not image_sent:
                user_content.append({"type": "image", "image": img})
                image_sent = True
            user_content.append({"type": "text", "text": str(user_text)})

            messages.append({"role": "user", "content": user_content})

            # ----- assistant reply -----
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": str(assistant_text)}],
            })

            turn += 1

        return {"messages": messages}

    def run(self, extra_train1=None, extra_test1=None, extra_train2=None, extra_test2=None):
        """
        Executes the fine-tuning process, including evaluation
        on 2-3 test samples at the end of each epoch.
        """
        # Convert your training and evaluation datasets
        converted_train_dataset = [self.format_data(row) for row in self.data]
        converted_eval_dataset  = [self.format_data(row) for row in self.eval_data]
        
        # --- optional add-ons -------------------------------------------
        if extra_train1 is not None:
            converted_train_dataset += [self.format_data_multiturn(r) for r in extra_train1]

        if extra_train2 is not None:
            converted_train_dataset += [self.format_data_multiturn(r) for r in extra_train2]

        if extra_test1 is not None:
            converted_eval_dataset += [self.format_data_multiturn(r) for r in extra_test1]

        if extra_test2 is not None:
            converted_eval_dataset += [self.format_data_multiturn(r) for r in extra_test2]
            
        
        # 3) TrainingArguments / SFTConfig
        training_args = SFTConfig(
            learning_rate=self.learning_rate,
            output_dir='./models/model_cot_qwen25vl_rerun_v1',
            optim=self.optim,
            logging_steps=1,
            report_to="none",
            
            # Use bf16 if available, else fallback to fp16
            fp16 = not is_bf16_supported(),
            bf16 = is_bf16_supported(),
            
            logging_first_step=True,
            warmup_ratio=self.warmup_ratio,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            logging_dir='./logs',
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            num_train_epochs=self.epochs,
            weight_decay = 0.01,            
            lr_scheduler_type = "linear",   
            seed = 3407,
            logging_strategy = "steps",
            
            # Evaluate at the end of every epoch
            # evaluation_strategy="epoch",
            
            # You MUST put the below items for vision finetuning:
            remove_unused_columns = False,
            dataset_text_field = None,
            dataset_kwargs = {"skip_prepare_dataset": True},
            dataset_num_proc = 4,
            max_seq_length = 2048,
        )
        
        # Model in training mode
        FastVisionModel.for_training(self.model)
        
        # 4) Create SFTTrainer with both train & eval sets
        trainer = SFTTrainer(
            model = self.model,
            tokenizer = self.tokenizer,
            data_collator = UnslothVisionDataCollator(self.model, self.tokenizer),
            train_dataset = converted_train_dataset,
            eval_dataset  = converted_eval_dataset,  # Evaluate on 2-3 items each epoch
            args = training_args,
            formatting_func = lambda x: x["messages"],
        )
        
        # 5) Start training. The trainer will evaluate at the end of each epoch
        trainer.train()


In [5]:
finetuner = FinetuneQwenVL(
    data=fine_tune_data,
    eval_data=eval_data,
    epochs=1,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    gradient_accumulation_steps=16,
    optim="adamw_torch_fused",
    model_id="unsloth/Qwen2.5-VL-7B-Instruct",
    peft_r=128,
    peft_alpha=128,
    peft_dropout=0.0,
)

finetuner.run(
    # extra_train1=circuit_train,
    # extra_test1=circuit_eval,
    # extra_train2=entangle_train,
    # extra_test2=entangle_eval,
)

==((====))==  Unsloth 2025.3.8: Fast Qwen2_5_Vl vision patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.684 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


NotImplementedError: Cannot copy out of meta tensor; no data!